In [1]:
import random
import pickle
from pathlib import Path
import numpy as np, wave
import pandas as pd
import torch
import torchaudio
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm   
from config import CHILD, CHILD_AFR_SADILAR, ADULT_NCHLT, NOISY_DIR, RESULTS_DIR
   
tqdm.pandas()    
#Fixed seeds and the 16 kHz rate Whisper expects
random.seed(42)
np.random.seed(42)
SAMPLE_RATE = 16000

In [2]:
from transformers import WhisperFeatureExtractor, WhisperModel

#Turns the audio into an 80-bin log-Mel spectrogram, padded or cut to 30 s
feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base")
#Only the encoder is used below, the decoder is ignored
model = WhisperModel.from_pretrained("openai/whisper-base")

device = "cpu"

#eval() switches off dropout so the embeddings are repeatable
lid_model = model.to(device).eval()
print(f"Model loaded on {device}")

/Users/conarj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Model loaded on cpu


In [3]:
#Takes n_total utterances spread as evenly as possible over the speakers
def balanced_subsample(df, n_total, group="speaker", seed=42):
    # rank utterances within each speaker (0,1,2,...) after shuffling,
    # then take the lowest ranks first -> one per speaker before a second, etc.
    shuffled = df.sample(frac=1, random_state=seed)
    rank = shuffled.groupby(group).cumcount()
    return (shuffled.assign(_r=rank.values)
                    .sort_values(["_r", group])
                    .head(n_total)
                    .drop(columns="_r")
                    .reset_index(drop=True))

In [4]:
#LOADING CHILD AFR DATASET:
CHILD_MANIFEST_AFR, CHILD_AUDIO_DIR_AFR = CHILD["afr"]

afr_child_df = pd.read_csv(CHILD_MANIFEST_AFR)
afr_child_df["path"] = afr_child_df["filename"].apply(lambda f: CHILD_AUDIO_DIR_AFR / f)
afr_child_df["true_lang"] = "afr"   
print(f"Loaded {len(afr_child_df)} AFR audio segments.") 

#Cut every language down to 10 085 utterances, the size of the Sesotho sa Leboa set
afr_child_df = balanced_subsample(afr_child_df, n_total=10085)  
print(afr_child_df["speaker"].value_counts().describe())
print(f"Afrikaans Child files subsampled : {len(afr_child_df)} / {len(afr_child_df)}")
afr_child_df["true_lang"] = "afr"

total_s = afr_child_df["duration_s"].sum()
print(f"Total duration of Afrikaans child dataset: {total_s:.2f} seconds, or {total_s/60:.2f} minutes, or {total_s/3600:.2f} hours.")

Loaded 13990 AFR audio segments.
count    154.000000
mean      65.487013
std       13.191519
min       26.000000
25%       60.000000
50%       73.000000
75%       74.000000
max       74.000000
Name: count, dtype: float64
Afrikaans Child files subsampled : 10085 / 10085
Total duration of Afrikaans child dataset: 13779.49 seconds, or 229.66 minutes, or 3.83 hours.


In [5]:
#LOADING CHILD ENGLISH DATASET : 
CHILD_MANIFEST_ENG, CHILD_AUDIO_DIR_ENG = CHILD["eng"]

eng_child_df = pd.read_csv(CHILD_MANIFEST_ENG)
#Each session has its own folder and the files are whole recordings, not cut segments
eng_child_df["path"] = eng_child_df.apply(lambda r: CHILD_AUDIO_DIR_ENG / r["session"] / r["filename"], axis=1)
#Speech duration from the time stamps, only used to report the hours
eng_child_df["duration_s"] = (eng_child_df["end_ms"] - eng_child_df["start_ms"]) / 1000.0

print(f"Loaded {len(eng_child_df)} ENG audio segments")

#No speaker column, so sessions are used as speakers
eng_child_df = balanced_subsample(eng_child_df, n_total=10085, group="session")   
print(eng_child_df["session"].value_counts().describe())
print(f"English Child files subsampled: {len(eng_child_df)} / {len(eng_child_df)}")
eng_child_df["true_lang"] = "eng"

total_s = eng_child_df["duration_s"].sum()
print(f"Total duration of English child dataset: {total_s:.2f} seconds, or {total_s/60:.2f} minutes, or {total_s/3600:.2f} hours.")

Loaded 13160 ENG audio segments
count     40.000000
mean     252.125000
std       36.101841
min       57.000000
25%      260.000000
50%      260.000000
75%      260.000000
max      260.000000
Name: count, dtype: float64
English Child files subsampled: 10085 / 10085
Total duration of English child dataset: 18178.68 seconds, or 302.98 minutes, or 5.05 hours.


In [6]:
#LOADING CHILD NORTHERN SOTHO DATASET : 
CHILD_MANIFEST_NSO, CHILD_AUDIO_DIR_NSO = CHILD["nso"]

nso_child_df = pd.read_csv(CHILD_MANIFEST_NSO)
nso_child_df["path"]       = nso_child_df["filename"].apply(lambda f: CHILD_AUDIO_DIR_NSO / f)
nso_child_df["duration_s"] = (nso_child_df["end_ms"] - nso_child_df["start_ms"]) / 1000.0

#Already 10 085 utterances, so no subsampling
print(f"Loaded {len(nso_child_df)} NSO audio segments")
nso_child_df["true_lang"] = "nso"

total_s = nso_child_df["duration_s"].sum()
print(f"Total duration of Sesotho sa Leboa child dataset: {total_s:.2f} seconds, or {total_s/60:.2f} minutes, or {total_s/3600:.2f} hours.")

Loaded 10085 NSO audio segments
Total duration of Sesotho sa Leboa child dataset: 28990.92 seconds, or 483.18 minutes, or 8.05 hours.


In [7]:
#LOADING CHILD ISIXHOSA DATASET
CHILD_MANIFEST_XHO, CHILD_AUDIO_DIR_XHO = CHILD["xho"]

xho_child_df = pd.read_csv(CHILD_MANIFEST_XHO)
xho_child_df["path"] = xho_child_df["filename"].apply(lambda f: CHILD_AUDIO_DIR_XHO / f)
xho_child_df["true_lang"] = "xho"       
print(f"Loaded {len(xho_child_df)} XHO audio segments.")

#Same subsampling as for Afrikaans
xho_child_df = balanced_subsample(xho_child_df, n_total=10085, group="speaker")   
print(xho_child_df["speaker"].value_counts().describe())
print(f"Xhosa Child files subsampled: {len(xho_child_df)} / {len(xho_child_df)}")
total_s = xho_child_df["duration_s"].sum()
print(f"Total duration of Xhosa child dataset: {total_s:.2f} seconds, or {total_s/60:.2f} minutes, or {total_s/3600:.2f} hours.")

Loaded 12947 XHO audio segments.
count    153.000000
mean      65.915033
std       18.706510
min        3.000000
25%       53.000000
50%       79.000000
75%       80.000000
max       80.000000
Name: count, dtype: float64
Xhosa Child files subsampled: 10085 / 10085
Total duration of Xhosa child dataset: 15011.88 seconds, or 250.20 minutes, or 4.17 hours.


In [8]:
#LOAD DUTCH DATASET
CHILD_MANIFEST_NLD, CHILD_AUDIO_DIR_NLD = CHILD["nld"]

nl_child_df = pd.read_csv(CHILD_MANIFEST_NLD)
nl_child_df["path"] = nl_child_df["filename"].apply(lambda f: CHILD_AUDIO_DIR_NLD / f)
nl_child_df["true_lang"] = "nl"        
print(f"Loaded {len(nl_child_df)} NL audio segments.")

#Same subsampling as for Afrikaans
nl_child_df = balanced_subsample(nl_child_df, n_total=10085, group="speaker")   
print(nl_child_df["speaker"].value_counts().describe())
print(f"Dutch Child files subsampled : {len(nl_child_df)} / {len(nl_child_df)}")
total_s = nl_child_df["duration_s"].sum()
print(f"Total duration of Dutch child dataset: {total_s:.2f} seconds, or {total_s/60:.2f} minutes, or {total_s/3600:.2f} hours.")

Loaded 16107 NL audio segments.
count    168.000000
mean      60.029762
std        3.550197
min       40.000000
25%       60.000000
50%       61.000000
75%       61.000000
max       61.000000
Name: count, dtype: float64
Dutch Child files subsampled : 10085 / 10085
Total duration of Dutch child dataset: 24656.68 seconds, or 410.94 minutes, or 6.85 hours.


In [9]:
#80/10/10 split per language, with no speaker in more than one set
#Same seeds and data as the MMS notebook, so the split is identical
train_dfs = []
test_dfs = []
val_dfs = []

#Afrikaans :
#First split off 20% of the speakers, then halve that into validation and test
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
afr_split = next(gss1.split(afr_child_df, groups=afr_child_df["speaker"]))
afr_train_df = afr_child_df.iloc[afr_split[0]].reset_index(drop=True)
afr_temp_test = afr_child_df.iloc[afr_split[1]].reset_index(drop=True)
#Split the test set into validation and test set
afr_val_split = next(gss2.split(afr_child_df.iloc[afr_split[1]], groups=afr_child_df.iloc[afr_split[1]]["speaker"]))
afr_val_df = afr_temp_test.iloc[afr_val_split[0]].reset_index(drop=True)
afr_test_df = afr_temp_test.iloc[afr_val_split[1]].reset_index(drop=True)

train_dfs.append(afr_train_df)
test_dfs.append(afr_test_df)
val_dfs.append(afr_val_df)

#TESTS: 
#Print all speakers in train and test set
afr_test_speakers = afr_test_df["speaker"].unique()
afr_train_speakers = afr_train_df["speaker"].unique()
afr_val_speakers = afr_val_df["speaker"].unique()

#Print number of total speakers in train and test set 
print("Afr test speakers: ", len(afr_test_speakers))
print("Afr train speakers: ", len(afr_train_speakers))  
print("Afr val speakers: ", len(afr_val_speakers))
print("Total speakers in both sets: ", len(afr_test_speakers) + len(afr_train_speakers) + len(afr_val_speakers))

#Check if any speakers are in both train and test set
for speaker in afr_test_speakers: 
    if speaker in afr_train_speakers: 
        print(f"Speaker {speaker} is in both train and test set for Afrikaans!")

#Check if any speakers are in both train and val set
for speaker in afr_val_speakers:
    if speaker in afr_train_speakers: 
        print(f"Speaker {speaker} is in both train and val set for Afrikaans!")

#Check if any speakers are in both test and val set
for speaker in afr_test_speakers:
    if speaker in afr_val_speakers: 
        print(f"Speaker {speaker} is in both test and val set for Afrikaans!")

#Print totals in train and test set
print("Afr train set total: ", len(afr_train_df))
print("Afr test set total: ", len(afr_test_df))
print("Afr val set total: ", len(afr_val_df))
print("Afr total: ", len(afr_train_df) + len(afr_test_df) + len(afr_val_df))

#English :
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
#Grouped by session, since the Nexdata manifest has no speaker column
eng_split = next(gss1.split(eng_child_df, groups=eng_child_df["session"]))
eng_train_df = eng_child_df.iloc[eng_split[0]].reset_index(drop=True)
eng_temp_test = eng_child_df.iloc[eng_split[1]].reset_index(drop=True)
#Split the test set into validation and test set
eng_val_split = next(gss2.split(eng_child_df.iloc[eng_split[1]], groups=eng_child_df.iloc[eng_split[1]]["session"]))
eng_val_df = eng_temp_test.iloc[eng_val_split[0]].reset_index(drop=True)
eng_test_df = eng_temp_test.iloc[eng_val_split[1]].reset_index(drop=True)

train_dfs.append(eng_train_df)
test_dfs.append(eng_test_df)
val_dfs.append(eng_val_df)

#Xhosa : 
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
xho_split = next(gss1.split(xho_child_df, groups=xho_child_df["speaker"]))
xho_train_df = xho_child_df.iloc[xho_split[0]].reset_index(drop=True)
xho_temp_test = xho_child_df.iloc[xho_split[1]].reset_index(drop=True)
#Split the test set into validation and test set
xho_val_split = next(gss2.split(xho_child_df.iloc[xho_split[1]], groups=xho_child_df.iloc[xho_split[1]]["speaker"]))
xho_val_df = xho_temp_test.iloc[xho_val_split[0]].reset_index(drop=True)
xho_test_df = xho_temp_test.iloc[xho_val_split[1]].reset_index(drop=True)

train_dfs.append(xho_train_df)
test_dfs.append(xho_test_df)
val_dfs.append(xho_val_df)

#Northern Sotho :
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
nso_split = next(gss1.split(nso_child_df, groups=nso_child_df["speaker"]))
nso_train_df = nso_child_df.iloc[nso_split[0]].reset_index(drop=True)
nso_temp_test = nso_child_df.iloc[nso_split[1]].reset_index(drop=True)
#Split the test set into validation and test set
nso_val_split = next(gss2.split(nso_child_df.iloc[nso_split[1]], groups=nso_child_df.iloc[nso_split[1]]["speaker"]))
nso_val_df = nso_temp_test.iloc[nso_val_split[0]].reset_index(drop=True)
nso_test_df = nso_temp_test.iloc[nso_val_split[1]].reset_index(drop=True)

train_dfs.append(nso_train_df)
test_dfs.append(nso_test_df)
val_dfs.append(nso_val_df)

#Dutch :
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
nl_split = next(gss1.split(nl_child_df, groups=nl_child_df["speaker"]))
nl_train_df = nl_child_df.iloc[nl_split[0]].reset_index(drop=True)
nl_temp_test = nl_child_df.iloc[nl_split[1]].reset_index(drop=True)
#Split the test set into validation and test set
nl_val_split = next(gss2.split(nl_child_df.iloc[nl_split[1]], groups=nl_child_df.iloc[nl_split[1]]["speaker"]))
nl_val_df = nl_temp_test.iloc[nl_val_split[0]].reset_index(drop=True)
nl_test_df = nl_temp_test.iloc[nl_val_split[1]].reset_index(drop=True)

train_dfs.append(nl_train_df)
test_dfs.append(nl_test_df)
val_dfs.append(nl_val_df)

#Create a single train, test and validation set by concatenating the individual language dataframes
train_df = pd.concat(train_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)
val_df = pd.concat(val_dfs, ignore_index=True)

print(f"Train set total: {len(train_df)}")
print(f"Test set total: {len(test_df)}")
print(f"Val set total: {len(val_df)}")

print(f"Train set language distribution:\n{train_df['true_lang'].value_counts()}")
print(f"Test set language distribution:\n{test_df['true_lang'].value_counts()}")
print(f"Val set language distribution:\n{val_df['true_lang'].value_counts()}")

print("Total wavs in all sets: ", len(train_df) + len(test_df) + len(val_df))

#Check that no speaker is in more than one set (English uses the session as speaker)
def speaker_ids(df):
    spk = df["session"].where(df["true_lang"] == "eng", df["speaker"])
    return set(zip(df["true_lang"], spk))

train_spk, val_spk, test_spk = speaker_ids(train_df), speaker_ids(val_df), speaker_ids(test_df)
assert not train_spk & val_spk,  "speaker in train and val"
assert not train_spk & test_spk, "speaker in train and test"
assert not val_spk & test_spk,   "speaker in val and test"

#Same supports as in the report
assert test_df["true_lang"].value_counts().to_dict() == {"nso": 1148, "afr": 1076, "xho": 1046, "eng": 1039, "nl": 1034}
assert val_df["true_lang"].value_counts().to_dict()  == {"eng": 1040, "xho": 1040, "nl": 1031, "nso": 974, "afr": 955}

Afr test speakers:  16
Afr train speakers:  123
Afr val speakers:  15
Total speakers in both sets:  154
Afr train set total:  8054
Afr test set total:  1076
Afr val set total:  955
Afr total:  10085
Train set total: 40042
Test set total: 5343
Val set total: 5040
Train set language distribution:
true_lang
afr    8054
nl     8020
eng    8006
xho    7999
nso    7963
Name: count, dtype: int64
Test set language distribution:
true_lang
nso    1148
afr    1076
xho    1046
eng    1039
nl     1034
Name: count, dtype: int64
Val set language distribution:
true_lang
eng    1040
xho    1040
nl     1031
nso     974
afr     955
Name: count, dtype: int64
Total wavs in all sets:  50425


In [ ]:
#Mean-pooled embedding of every encoder layer for one file
@torch.no_grad()
def extract_all_layers(audio_path, model=lid_model, feature_extractor=feature_extractor, device=device) : 
    waveform, sr = torchaudio.load(audio_path)
    if sr != SAMPLE_RATE: 
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    audio = waveform.squeeze().numpy()
    n_samples = audio.shape[-1]
    
    inputs = feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(device)
    enc = model.encoder(inputs["input_features"], output_hidden_states=True)   # ENCODER only, keep all hidden states
    #Every clip is padded to 30 s (1 500 frames, 50 per second), so only average the frames with real audio
    valid = int(round(n_samples / SAMPLE_RATE * 50))
    #At least one frame, and at most 1 500 for clips longer than 30 s
    valid = max(1, min(valid, enc.last_hidden_state.shape[1]))
    pooled_layers = [] 
    #hidden_states[0] is the input to the first encoder layer, hidden_states[l] the output of layer l
    for h in enc.hidden_states: 
        real = h[:, :valid, :]
        avg = real.mean(dim=1) 
        avg = avg.squeeze()
        pooled_layers.append(avg)
    
    result = torch.stack(pooled_layers)       
    return result.cpu().numpy()

In [ ]:
#EXTRACT EMBEDDINGS FOR ALL LAYERS OF TRAIN, TEST AND VAL SETS.
#Slow, so each set is saved to a pickle straight away
train_df["all_layers"] = train_df["path"].progress_apply(extract_all_layers)
#train_df.to_pickle("training_set_with_all_layer_emb_whisper.pkl") #Save the embeddings

test_df["all_layers"] = test_df["path"].progress_apply(extract_all_layers)
#test_df.to_pickle("test_set_with_all_layer_emb_whisper.pkl") #Save the embeddings

val_df["all_layers"] = val_df["path"].progress_apply(extract_all_layers)
#val_df.to_pickle("val_set_with_all_layer_emb_whisper.pkl") #Save the embeddings

In [ ]:
L = train_df["all_layers"].values[0].shape[0] #number of layers 
print(train_df["all_layers"].values[0].shape)
layer_scores = []
#One classifier per layer, scored on the validation set only
for l in range(L) : 
    train_layer_embeddings = []
    val_layer_embeddings = []
    for n in range(len(train_df)) :
        train_clip_embeddings = train_df["all_layers"].values[n]
        train_layer_embeddings.append(train_clip_embeddings[l])
    
    for n in range(len(val_df)) : 
        val_clip_embeddings = val_df["all_layers"].values[n]
        val_layer_embeddings.append(val_clip_embeddings[l])
    
    X_train = np.vstack(train_layer_embeddings)
    y_train = train_df["true_lang"].values

    X_val = np.vstack(val_layer_embeddings)
    y_val = val_df["true_lang"].values

    print("Training shape", l, X_train.shape)
    print("Validation shape", l, X_val.shape)

    #Scaler and classifier in one pipeline, so the scaler is only fit on the training data
    clf  = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    layer_scores.append(classification_report(val_df["true_lang"].values, y_pred, output_dict=True))

#Validation accuracy and F1 per layer, printed as LaTeX table rows
for l, rep in enumerate(layer_scores):
    f1 = {c: rep[c]["f1-score"] for c in ("afr", "eng", "nl" "nso", "xho")}
    print(f"{l:>2} & {rep['accuracy']:.3f} & {f1['afr']:.2f} & {f1['eng']:.2f} & {f1['nl']:.2f}  "
          f"& {f1['nso']:.2f} & {f1['xho']:.2f} \\\\")

In [ ]:
#Rerun classifier on best performing layer : 
#Layer 1 is best on validation. The test set is only used here
train_layer_embeddings = [] 
test_layer_embeddings = [] 
for n in range(len(train_df)): 
    train_clip_embeddings = train_df["all_layers"].values[n]
    train_layer_embeddings.append(train_clip_embeddings[1])

for n in range(len(test_df)): 
    test_clip_embeddings = test_df["all_layers"].values[n]
    test_layer_embeddings.append(test_clip_embeddings[1])

X_train = np.vstack(train_layer_embeddings)
y_train = train_df["true_lang"].values

X_test = np.vstack(test_layer_embeddings)
y_test = test_df["true_lang"].values

print("Training shape", X_train.shape)
print("Test shape", X_test.shape)

clf  = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)


print(classification_report(y_test, y_pred))

In [ ]:
#Visualise the confusion matrix
#Rows are the true languages, columns the predicted ones
cm = confusion_matrix(test_df["true_lang"].values, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot(cmap=plt.cm.Blues)